# ROGII — Wellbore Geology Prediction
## Notebook 0: Competition Context & Domain Knowledge

**Competition**: [Kaggle — ROGII Wellbore Geology Prediction](https://www.kaggle.com/competitions/rogii-wellbore-geology-prediction)  
**Metric**: RMSE on True Vertical Thickness (TVT)  
**Type**: Kernel-only, code runs on Kaggle at submission time

## 1. Problem Overview

### What is Geosteering?

Drilling a horizontal oil/gas well means steering the drill bit through a **thin reservoir layer** (often 5–30 m thick). This is called **geosteering**. Traditional geosteering relies on geologists manually interpreting gamma-ray logs in real time — which is slow and subjective.

ROGII builds geosteering software (StarSteer). This competition challenges ML to **automate geological interpretation** by predicting **True Vertical Thickness (TVT)** along the horizontal wellbore.

### What is TVT?

TVT encodes **where the wellbore is within the geological layer stack** — specifically the vertical position of the drill bit relative to formation boundaries.

```
Surface
  │  (vertical section)
  └────────────────────────────── horizontal section ──────────────────
                │   SHALE (high GR)                                    │
      ┌─────────┼──────────────────────────────────────────────────────┤
      │ RESERVOIR│ ←── drill bit here                                  │
      │  (low GR)│   TVT = how far are we from the top of the layer?   │
      └──────────┼──────────────────────────────────────────────────────┘
                │   SHALE (high GR)                                    │
```

Predicting TVT accurately tells the driller: *"You are 8 m below the formation top — steer slightly upward."*

## 2. Dataset Structure

```
data/
  train/
    {well_id}__horizontal_well.csv   # 773 wells — full labels
    {well_id}__typewell.csv          # 773 reference wells
    {well_id}.png                    # visualization (not used in ML)
  test/
    {well_id}__horizontal_well.csv   # 3 wells — TVT hidden
    {well_id}__typewell.csv          # 3 reference wells
  sample_submission.csv
  AI_wellbore_geology_prediction_task_en.pptx
```

Well IDs are 8-character hex strings (e.g. `000d7d20`). The 3 test wells **also appear in the train set** — the competition hides specific rows' TVT and asks us to predict them.

### Horizontal Well CSV — columns

| Column | Description | Available in test? |
|---|---|---|
| `MD` | Measured depth along wellbore (m) | ✓ |
| `X`, `Y`, `Z` | 3D trajectory coordinates (m) | ✓ |
| `GR` | Gamma-Ray log — low = reservoir, high = shale | ✓ |
| `TVT_input` | Partially-known TVT (expert interpretation; 0 where unknown) | ✓ |
| `ANCC` | Resistivity — attenuation near-compensated | ✗ train only |
| `ASTNU` | Resistivity — attenuation shallow near upper | ✗ train only |
| `ASTNL` | Resistivity — attenuation shallow near lower | ✗ train only |
| `EGFDU` | Resistivity — phase shift far upper | ✗ train only |
| `EGFDL` | Resistivity — phase shift far lower | ✗ train only |
| `BUDA` | Resistivity — ? | ✗ train only |
| `TVT` | **TARGET** — true vertical thickness | ✗ (what we predict) |

### Typewell CSV — columns

| Column | Description |
|---|---|
| `TVT` | Depth axis in the reference vertical well |
| `GR` | Gamma-Ray of the reference well at that TVT |
| `Geology` | Formation label (mostly empty) |

### Sample Submission format

```
id,tvt
000d7d20_1442,12345.67
```

ID = `{well_id}_{row_index}` (0-based index within the test file).

## 3. The Core Signal: GR vs TVT

```
GR (API units)
│
│  150 │████████████████│  SHALE ABOVE  (high GR)
│      │                │
│   50 │  ░░░░░░░░░░░░  │  RESERVOIR    (low GR) ← drill target
│      │                │
│  150 │████████████████│  SHALE BELOW  (high GR)
│
└─────────────────────────────────▶ TVT (position in layer stack)
```

The GR log **encodes position within the layer**. As TVT increases (moving down), GR transitions: low → high (exiting the reservoir top into the shale above). As TVT decreases, GR goes high → low (entering the reservoir from below).

**Key difficulty**: GR drifts between wells (different borehole conditions, tool calibration). Normalization relative to the typewell is critical.

## 4. TVT_input: The Most Powerful Feature

`TVT_input` is a partially-observed version of the target. For rows where a geologist has already interpreted the geology, `TVT_input = TVT`. For rows to predict, `TVT_input = 0`.

This means the problem is essentially **interpolation/extrapolation**: given known TVT anchors and the continuous GR signal, fill in the unknown TVT values.

A strong baseline is just **linear interpolation** between known `TVT_input` anchors — the ML model's job is to improve on this by using the GR signal to detect layer boundaries more precisely.

## 5. Typewell: The Reference Fingerprint

The typewell provides the **expected GR pattern** for this geological area as a function of TVT. It's the "template" that the horizontal well should match.

By cross-correlating the horizontal GR with the typewell GR (at different TVT offsets), we can directly estimate where in the formation the wellbore is — this is the closest technique to how real geosteering software works.

```
Typewell GR vs TVT:      Horizontal well GR:

TVT=11230 → GR=126       MD=11467 → GR=115  ← matches TVT≈11236 in typewell
TVT=11235 → GR=115       MD=11468 → GR=116
TVT=11240 → GR=95        MD=11469 → GR=135  ← GR rising → moving toward shale
```

## 6. Resistivity Logs (Train Only)

The 6 resistivity measurements (`ANCC, ASTNU, ASTNL, EGFDU, EGFDL, BUDA`) are electromagnetic measurements at different depths-of-investigation. They distinguish formation fluids (water vs oil) and lithology independently of GR.

**These are not available in the test set**, so they cannot be used as direct model inputs. However:
- They can be used as **auxiliary targets** during training (multi-task learning)
- They can help build better features during the training phase only
- They might improve typewell alignment logic

## 7. Leaderboard Landscape

| Approach | LB RMSE | Key technique |
|---|---|---|
| Linear interpolation of TVT_input | ~15 | Baseline |
| LightGBM on GR features | ~12 | Rolling stats, lags |
| + Typewell GR alignment | ~10 | Cross-correlation feature |
| + DWT on GR | ~9.25 | Wavelet decomposition |
| Advanced ensemble | ~9.96 | Multi-model blend |
| Top tier | <8 | Physics-informed + LSTM |

## 8. Roadmap

```
01_eda.ipynb       → understand the 773 wells, GR distributions,
                     TVT_input masking patterns, typewell alignment

02_modeling.ipynb  → feature engineering + LightGBM + XGBoost + CatBoost
                     ensemble → submission

03_advanced.ipynb  → Optuna tuning, BiLSTM, particle filter,
                     typewell cross-correlation, pseudo-labeling
```